# BrickOrSwish: video → detections.json + annotated video

Runs on Google's machine, not yours. Steps: upload a video, click the rim 3 times, then COCO-pretrained
**RF-DETR Nano** (Apache-2.0) detects the ball on every frame. You get:

| File | What |
|---|---|
| `hoop.json` | your 3 clicks (`shared/schema/hoop.schema.json`) |
| `camera_motion.json` | where the hoop was found every 0.5 s: when the camera moved, and spans where the hoop was lost (not counted) |
| `detections.json` | per-frame boxes (`shared/schema/detections.schema.json`) in the pixels of the frame you tapped, with camera motion removed; ball is `"ball"` |
| `events.json` | shots found by our Python shot logic (`shared/schema/events.schema.json`) |
| `labels.json`, `score.json` | your verdict per shot and the resulting accuracy (after the verdict cell) |
| `annotated.mp4` | "camera moved" / "HOOP NOT FOUND" notes, boxes (orange = ball the logic uses, dark orange = ball it ignores), hoop ellipse, approach zone, tracked ball (green), state and make/miss |

**Security:** the setup installs pinned versions (`ml/requirements-colab.txt` once it exists), then the audit cell
checks PyTorch is new enough and checks every installed dependency against OSV.dev. It stops if anything is vulnerable.

**First:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all, or run the cells one by one.
An uploaded video lives only on this temporary machine and is gone when the runtime is disconnected.
**Same video again?** Put it in your Google Drive once and set `VIDEO_PATH` in Settings: no upload, and paste the
printed hoop into `HOOP` to skip clicking too.

In [ ]:
# Settings
BRANCH = "feat/shot-logic-reference"  # branch of github.com/Pietrzaaq/swish-or-brick to run
RFDETR_VERSION = "1.12.1"             # used only until ml/requirements-colab.txt (all versions pinned) exists
ALLOW_VULNS = False                   # True = continue even if the security audit finds known vulnerabilities
VIDEO_PATH = ""     # skip the upload: a video in your Google Drive, e.g. "/content/drive/MyDrive/brickorswish/IMG_0017.MP4"
                    # (or "/content/IMG_0017.MP4" if you uploaded it earlier and this runtime is still connected)
CROP = "hoop"       # "hoop": detector sees a square around the rim (the ball is bigger); "full": whole frame
THRESHOLD = 0.15    # keep low: the shot logic filters at ball_min_conf (0.3) itself
HOOP_AT_S = 0.0     # second of the video to tap the rim on: pick a moment when the phone is already steady
STABILIZE = True    # follow the hoop if the camera moves after you tapped it (see camera_motion.json)
HOOP = None         # paste a hoop.json dict here to skip clicking, e.g. {"image_width": 1920, ...}

In [ ]:
# Setup: code + detector library (1-3 minutes)
import os, sys
!nvidia-smi -L || echo "NO GPU: Runtime > Change runtime type > T4 GPU (CPU works but is very slow)"
!rm -rf /content/swish-or-brick && git clone -q --depth 1 -b {BRANCH} https://github.com/Pietrzaaq/swish-or-brick.git /content/swish-or-brick
!git -C /content/swish-or-brick log -1 --format="running code from commit %h: %s (%cr)"
sys.path.insert(0, "/content/swish-or-brick/ml")
if not os.path.exists("/content/swish-or-brick/ml/brickorswish/detect/stabilize.py"):
    raise RuntimeError(f"{BRANCH} on GitHub has no camera-motion code yet: push your commits, then Run all again")
PINS = "/content/swish-or-brick/ml/requirements-colab.txt"
if os.path.exists(PINS):
    !pip install -q -r {PINS}
else:
    print("No pinned versions in the repo yet: installing rfdetr only. The audit cell writes the pin file.")
    !pip install -q rfdetr=={RFDETR_VERSION}

import subprocess
def run_py(*args):
    # python -m <args> in a fresh process, streaming its output. Colab preloads some libraries (e.g. an old
    # Pillow) into this notebook's process at startup; a fresh process loads only what is installed now.
    proc = subprocess.Popen([sys.executable, "-u", "-m", *map(str, args)], cwd="/content/swish-or-brick/ml",
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    return proc.wait()

In [ ]:
# Security audit
# 1. PyTorch >= 2.10: older versions let a crafted .pth file run code even in safe mode (CVE-2026-24747).
# 2. Every installed package the detector depends on, at its exact version, checked against OSV.dev
#    (CVE / GitHub / PyPI advisories and known malicious packages). Vulnerable ones are upgraded to the
#    smallest fixed release and checked again. Writes /content/requirements-colab.txt.
# Runs in a fresh process (run_py), so it checks exactly what the detector will load.
from importlib.metadata import version
from brickorswish.tools.deps_audit import version_tuple

if version_tuple(version("torch")) < (2, 10):
    print(f"torch {version('torch')} is older than 2.10, upgrading")
    !pip install -q -U "torch>=2.10" torchvision
assert version_tuple(version("torch")) >= (2, 10), f"torch {version('torch')} is still older than 2.10"
print(f"torch {version('torch')} OK")

ROOTS = ["rfdetr", "opencv-python", "opencv-python-headless", "opencv-contrib-python"]  # missing ones are skipped
rc = run_py("brickorswish.tools.deps_audit", *ROOTS, "--pins", "/content/requirements-colab.txt", "--fix")
if rc != 0 and not ALLOW_VULNS:
    raise RuntimeError("Audit failed (see above). Stop here, or set ALLOW_VULNS = True if you accept the risk.")
print("audit clean")

In [ ]:
# The video: from VIDEO_PATH (Google Drive or this runtime) if set, otherwise upload it
import shutil
from google.colab import files

if VIDEO_PATH:
    if VIDEO_PATH.startswith("/content/drive/"):
        from google.colab import drive
        drive.mount("/content/drive")  # asks you to allow access the first time
    if not os.path.exists(VIDEO_PATH):
        raise FileNotFoundError(f"{VIDEO_PATH} not found: check the path in Settings (Files panel on the left)")
    VIDEO = VIDEO_PATH
    if VIDEO.startswith("/content/drive/"):
        # local copy: the video is read 3 times (camera motion, detection, drawing), Drive reads are slower
        VIDEO = "/content/" + os.path.basename(VIDEO_PATH)
        if not os.path.exists(VIDEO):
            shutil.copy(VIDEO_PATH, VIDEO)
else:
    uploaded = files.upload()
    VIDEO = "/content/" + next(iter(uploaded))
print(VIDEO)

In [ ]:
# Hoop calibration: click LEFT rim edge, RIGHT rim edge, then the FRONT edge (nearest the camera)
import json
from brickorswish.detect.colab import pick_hoop
from brickorswish.detect.video import read_frame

import cv2
_cap = cv2.VideoCapture(VIDEO)
FPS = _cap.get(cv2.CAP_PROP_FPS) or 30.0
_cap.release()
HOOP_FRAME = int(round(HOOP_AT_S * FPS))
frame = read_frame(VIDEO, HOOP_FRAME)
print(f"tapping on frame {HOOP_FRAME} ({HOOP_AT_S:.1f} s)")
hoop = HOOP or pick_hoop(frame)
print(json.dumps(hoop))  # copy this into HOOP above to reuse it

In [ ]:
# Check: yellow ellipse should sit on the rim; grey square = what the detector sees
import cv2
from IPython.display import Image, display
from brickorswish.detect.annotate import draw_hoop_preview
from brickorswish.detect.crop import watch_crop
from brickorswish.logic import ShotLogicConfig
from brickorswish.paths import CONFIG_V1

cfg = ShotLogicConfig.load(CONFIG_V1)
preview = draw_hoop_preview(frame, hoop, cfg, watch_crop(hoop, cfg) if CROP == "hoop" else None)
scale = 900 / max(preview.shape[:2])
small = cv2.resize(preview, None, fx=scale, fy=scale) if scale < 1 else preview
display(Image(data=cv2.imencode(".jpg", small)[1].tobytes()))  # no Pillow in this process

In [ ]:
# Run: detection (a few minutes per minute of 1080p video on a T4), shot logic, annotated video
# Runs in a fresh process (run_py) with the audited library versions.
from pathlib import Path

OUT = Path("/content/out") / Path(VIDEO).stem
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "hoop.json").write_text(json.dumps(hoop))
if run_py("brickorswish.detect.run_video", VIDEO, "--hoop", OUT / "hoop.json", "--out", OUT,
          "--crop", CROP, "--threshold", THRESHOLD, "--hoop-frame", HOOP_FRAME,
          *([] if STABILIZE else ["--no-stabilize"])) != 0:
    raise RuntimeError("Run failed, see the error above.")

In [ ]:
# Watch the annotated video here (skipped above 50 MB; download it instead)
import base64
from IPython.display import HTML, display

mp4 = OUT / "annotated.mp4"
if mp4.stat().st_size < 50e6:
    b64 = base64.b64encode(mp4.read_bytes()).decode()
    display(HTML(f'<video controls style="max-width:100%" src="data:video/mp4;base64,{b64}"></video>'))
else:
    print(f"{mp4.stat().st_size / 1e6:.0f} MB, download it with the next cell")

In [ ]:
# Your verdict: after watching, type the TRUE outcome of every shot the app detected, in order:
#   O = make, X = miss, - = not a real shot (the app saw a shot that wasn't one)
# MISSED: real attempts the app did not list at all, e.g. "X" for one undetected miss.
# Then run this cell again: it prints accuracy against the Milestone 1 targets and saves labels.json + score.json.
TRUTH = ""    # e.g. "XOOOOOO"
MISSED = ""

from brickorswish.eval.score import format_report, save, score

events = json.loads((OUT / "events.json").read_text())
if not TRUTH:
    for s in events["shots"]:
        print(f"  shot {s['shot']:>2} at {s['t_start']:6.1f}s: {s['outcome']}")
    print(f"Type {len(events['shots'])} letters into TRUTH at the top of this cell, then run it again.")
else:
    result = score(events, TRUTH, MISSED)
    save(OUT, result)
    print(format_report(result))

In [ ]:
# Download everything as one zip (also contains requirements-colab.txt: commit it as ml/requirements-colab.txt
# so the next runs install exactly these versions)
import shutil
shutil.copy("/content/requirements-colab.txt", OUT / "requirements-colab.txt")
zip_path = shutil.make_archive(str(OUT), "zip", OUT)
files.download(zip_path)